In [ ]:
# When It Fails: Generalization in Pictures, and Inductive Bias
# Generated from the canonical HTML manuscript. Run this cell first.
# Source: https://github.com/Shakeri-Lab/dl-book/blob/c058d1f401fd0ead3ae59a2a8730f95489a2d9aa/chapters/part1/06-generalization-inductive-bias.qmd

from importlib.metadata import PackageNotFoundError, version as package_version
import hashlib as _bootstrap_hashlib
import os as _bootstrap_os
from pathlib import Path as _BootstrapPath
import subprocess as _bootstrap_subprocess
import sys as _bootstrap_sys
import urllib.request as _bootstrap_urlrequest

_BOOK_REVISION = 'c058d1f401fd0ead3ae59a2a8730f95489a2d9aa'
_PINNED_REQUIREMENTS = [
    "torch==2.12.1",
    "torchvision==0.27.1",
    "numpy==2.5.1",
    "matplotlib==3.11.1"
]
_BOOK_ASSETS = [
    {
        "path": "code/dlbook/__init__.py",
        "sha256": "5f31ed4ff3aac6a557697078bfc7b9048811de3c1ccc0dc19890a6b1499ffb06"
    },
    {
        "path": "code/dlbook/supervised.py",
        "sha256": "2e035d607b3e6771bd257dd700ee271e9be6b1b1fe6a4122c62b672ef0a266ad"
    },
    {
        "path": "data/fashion-test.pt",
        "sha256": "1db79d080c51173c7df18a5e8389dd4ae20ecb0352a21be90aaa446aed612a09"
    },
    {
        "path": "data/fashion-train.pt",
        "sha256": "86a99167f14d98891de2bc34b83197727f89e178cdf9e5b019fceb7bf71cc427"
    }
]

def _installed_requirement(requirement: str) -> bool:
    name, expected = requirement.split('==', 1)
    try:
        return package_version(name) == expected
    except PackageNotFoundError:
        return False

_missing_requirements = [
    item for item in _PINNED_REQUIREMENTS if not _installed_requirement(item)
]
if _missing_requirements:
    _bootstrap_install = _bootstrap_subprocess.run(
        [_bootstrap_sys.executable, '-m', 'pip', 'install', '--quiet',
         *_missing_requirements],
        check=False, capture_output=True, text=True,
    )
    if _bootstrap_install.returncode != 0:
        raise RuntimeError(_bootstrap_install.stdout + _bootstrap_install.stderr)

_bootstrap_base = _BootstrapPath(
    _bootstrap_os.environ.get(
        'DLBOOK_NOTEBOOK_ROOT',
        '/content' if _BootstrapPath('/content').is_dir()
        else str(_BootstrapPath.home() / '.cache'),
    )
)
_BOOK_ROOT = _bootstrap_base / f'dl-book-{_BOOK_REVISION[:12]}'
_RAW_ROOT = 'https://raw.githubusercontent.com/Shakeri-Lab/dl-book/' + _BOOK_REVISION + '/'
for _record in _BOOK_ASSETS:
    _destination = _BOOK_ROOT / _record['path']
    _destination.parent.mkdir(parents=True, exist_ok=True)
    _valid = (
        _destination.is_file()
        and _bootstrap_hashlib.sha256(_destination.read_bytes()).hexdigest()
        == _record['sha256']
    )
    if not _valid:
        _temporary = _destination.with_suffix(_destination.suffix + '.part')
        _bootstrap_urlrequest.urlretrieve(_RAW_ROOT + _record['path'], _temporary)
        _digest = _bootstrap_hashlib.sha256(_temporary.read_bytes()).hexdigest()
        if _digest != _record['sha256']:
            _temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Checksum mismatch for {_record['path']}")
        _temporary.replace(_destination)

(_BOOK_ROOT / 'chapters/part1').mkdir(parents=True, exist_ok=True)
_bootstrap_sys.path.insert(0, str(_BOOK_ROOT / 'code'))
_bootstrap_os.chdir(_BOOK_ROOT / 'chapters/part1')

# Hidden manuscript support required by later learner-visible cells.
# Plot-only harnesses are not exported.
import torch
from torch import nn

assert _BOOK_ROOT.is_dir()

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Define the reusable helpers: `train_mlp` and `accuracy`.
3. Fit the Chapter 4 recipe on the fixed Fashion-MNIST development split.
4. Report or visualize the measured result.

In [ ]:
import torch
from torch import nn

# [1]
torch.manual_seed(6050)
development = torch.load("../../data/fashion-train.pt")
holdout = torch.load("../../data/fashion-test.pt")
X_dev = development["X"].float().reshape(-1, 784) / 255.0  # (1200, 784)
y_dev, classes = development["y"], development["classes"]
X_test = holdout["X"].float().reshape(-1, 784) / 255.0     # (600, 784)
y_test = holdout["y"]

split = torch.randperm(len(X_dev), generator=torch.Generator().manual_seed(6050))
fit_idx, val_idx = split[:1000], split[1000:]
X_tr, y_tr = X_dev[fit_idx], y_dev[fit_idx]                 # (1000, 784), (1000,)
X_val, y_val = X_dev[val_idx], y_dev[val_idx]               # (200, 784), (200,)

from dlbook.supervised import fit_supervised   # Listing 4.1

# [2]
def train_mlp(
    X: torch.Tensor, y: torch.Tensor, hidden: int = 256,
    epochs: int = 40, seed: int = 6050
) -> nn.Module:
    # The delta from Listing 4.1 is one line: this chapter's model.
    return fit_supervised(
        lambda: nn.Sequential(
            nn.Linear(784, hidden), nn.ReLU(), nn.Linear(hidden, 10)
        ),
        X, y, epochs=epochs, seed=seed,
    )

def accuracy(net: nn.Module, X: torch.Tensor, y: torch.Tensor) -> float:
    net.eval()
    with torch.no_grad():
        return (net(X).argmax(1) == y).float().mean().item()

# [3]
net = train_mlp(X_tr, y_tr)
# [4]
print(f"train accuracy: {accuracy(net, X_tr, y_tr):.1%}")
print(f"validation accuracy: {accuracy(net, X_val, y_val):.1%}")

**Plan**

1. Define the reusable `shift_right` helper.
2. Shift validation images while keeping the trained MLP fixed.

In [ ]:
# [1]
def shift_right(X: torch.Tensor, px: int) -> torch.Tensor:
    img = X.reshape(-1, 28, 28)
    out = torch.zeros_like(img)
    if px > 0:
        out[:, :, px:] = img[:, :, :-px]
    else:
        out = img.clone()
    return out.reshape(-1, 784)

# [2]
for px in [0, 2]:
    print(f"shift {px}px: validation accuracy "
          f"{accuracy(net, shift_right(X_val, px), y_val):.1%}")

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Plot accuracy vs. shift.

In [ ]:
import matplotlib.pyplot as plt

# [1]
shifts = list(range(5))
accs = [accuracy(net, shift_right(X_val, s), y_val) for s in shifts]
plt.figure(figsize=(5.6, 3.3))
# [2]
plt.plot(shifts, accs, "o-", color="#E57200", lw=2, ms=7)
plt.axhline(0.1, ls=":", color="#B8B8A8")
plt.text(0.1, 0.115, "chance (10 classes)", color="#8A8A7A", fontsize=9)
plt.xlabel("shift (pixels right)"); plt.ylabel("validation accuracy")
plt.ylim(0, 0.9); plt.xticks(shifts)
plt.tight_layout(); plt.show()

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Retrain on one fixed permutation of the pixel coordinates.
3. Report or visualize the measured result.

In [ ]:
# [1]
torch.manual_seed(0)
pixel_perm = torch.randperm(784)          # one fixed scrambling for all images

# [2]
net_shuffled = train_mlp(X_tr[:, pixel_perm], y_tr)
# [3]
print(f"original images:  validation accuracy {accuracy(net, X_val, y_val):.1%}")
print(f"shuffled pixels:  validation accuracy "
      f"{accuracy(net_shuffled, X_val[:, pixel_perm], y_val):.1%}")

**Plan**

1. Select the largest-norm first-layer templates.

In [ ]:
# [1]
W1 = net[0].weight.detach()               # (256, 784)
order = W1.norm(dim=1).argsort(descending=True)[:16]
scale = float(W1[order].abs().max())

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Measure validation error across widths and fitting-set sizes.

In [ ]:
# [1]
widths = [2, 4, 8, 16, 64, 256]
# [2]
width_errors = {
    n: [1 - accuracy(train_mlp(X_tr[:n], y_tr[:n], hidden=h, epochs=60),
                       X_val, y_val)
        for h in widths]
    for n in (300, 1000)
}

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Measure the nearest-to-farthest distance ratio by dimension.

In [ ]:
# [1]
torch.manual_seed(6050)
dims = [2, 10, 100, 784, 5000]
ratios = []
# [2]
for d in dims:
    P = torch.rand(300, d)
    D = torch.cdist(P, P)
    ratios.append(((D + torch.eye(300) * 1e9).min(1).values / D.max(1).values).mean())

**Plan**

1. Run one final test-set check.
2. Report or visualize the measured result.

In [ ]:
# [1]
net_final = train_mlp(X_dev, y_dev)
# [2]
print(f"clean test accuracy:   {accuracy(net_final, X_test, y_test):.1%}")
print(f"shift-2 test accuracy: "
      f"{accuracy(net_final, shift_right(X_test, 2), y_test):.1%}")